# **Libraries**

In [2]:
# Analysis 
import numpy as np
import pandas as pd
import plotly.express as px 

# Adjusting Plotly Graphs
import plotly.io as pio
pio.renderers.default = "iframe"

# Preprocessing 
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer 
from sklearn.model_selection import train_test_split , StratifiedKFold , RandomizedSearchCV , cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.inspection import permutation_importance

# Classification Models 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation 
from sklearn.metrics import classification_report, confusion_matrix

# Saving Models
import joblib as jl

In [3]:
import warnings
warnings.filterwarnings('ignore')

# **Loading Data**

In [4]:
df = pd.read_csv("/kaggle/input/datasets/sushant097/bank-marketing-dataset-full/bank-full.csv", sep= ';')
df.sample(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
41998,55,technician,married,secondary,no,0,yes,yes,cellular,27,oct,88,1,-1,0,unknown,no
13056,26,admin.,single,secondary,no,-378,no,yes,cellular,8,jul,165,1,-1,0,unknown,no
8324,33,technician,single,tertiary,no,5234,no,no,unknown,2,jun,213,3,-1,0,unknown,no
21975,49,housemaid,married,secondary,no,3,no,no,cellular,20,aug,118,2,-1,0,unknown,no
32345,40,technician,married,primary,no,644,yes,no,cellular,16,apr,118,2,336,1,failure,no


# **Handling Data**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [6]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [7]:
df.describe(exclude=np.number)

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


In [8]:
df.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

# **EDA**

In [10]:
fig1 = px.pie(df,
              names='y',
              title='Target Distribution',
              hole= 0.4,
              color_discrete_sequence=['#EF553B', '#00CC96'],
              template='plotly_dark').show()

In [11]:
Job = df.groupby(['job', 'y']).size().reset_index(name='count')
fig2 = px.bar(Job,
              x= 'job',
              y= 'count',
              color= 'y',
              barmode= 'group',
              title= "Distribution of Job by Subscription Status",
              color_discrete_map= {'no': '#EF553B', 'yes': '#00CC96'},
              labels= {'job' : 'Job', 'count' : 'Count', 'y' : 'Status'},
              template= 'plotly_dark').show()

In [12]:
Education = df.groupby(['education', 'y']).size().reset_index(name='count')
fig3 = px.bar(Education,
              x= 'education',
              y= 'count',
              color= 'y',
              barmode= 'group',
              title= f"Distribution of Education by Subscription Status",
              color_discrete_map= {'no': '#EF553B', 'yes': '#00CC96'},
              labels= {'education' : 'Education', 'count' : 'Count', 'y' : 'Status'},
              template= 'plotly_dark').show()

In [13]:
Poutcome = df.groupby(['poutcome', 'y']).size().reset_index(name='count')
fig4 = px.bar(Poutcome,
              x= 'poutcome',
              y= 'count',
              color= 'y',
              barmode= 'group',
              title= "Distribution of Poutcome by Subscription Status",
              color_discrete_map= {'no': '#EF553B', 'yes': '#00CC96'},
              labels= {'poutcome' : 'Poutcome', 'count' : 'Count', 'y' : 'Status'},
              template= 'plotly_dark').show()

In [14]:
Age = df.groupby(['age', 'y']).size().reset_index(name= 'count')

fig5 = px.scatter(Age,
                  x= 'age',
                  y='count',
                  color= 'y',
                  title= f"Distribution of Age by Subscribtion Status",
                  labels= {'age' : 'Age' , 'count' : 'Count' , 'y' : 'Status'},
                  color_discrete_map={'no': '#EF553B', 'yes': '#00CC96'},
                  marginal_x= 'box',
                  template= 'plotly_dark').show()

In [15]:
Balance = df.groupby(['balance', 'y']).size().reset_index(name= 'count')

fig6 = px.scatter(Balance,
                  x= 'balance',
                  y='count',
                  color= 'y',
                  title= f"Distribution of Balance by Subscribtion Status",
                  labels= {'balance' : 'Balance' , 'count' : 'Count' , 'y' : 'Status'},
                  color_discrete_map={'no': '#EF553B', 'yes': '#00CC96'},
                  marginal_x= 'box',
                  template= 'plotly_dark').show()

In [16]:
Duration = df.groupby(['duration', 'y']).size().reset_index(name= 'count')

fig7 = px.scatter(Duration,
                  x= 'duration',
                  y='count',
                  color= 'y',
                  title= f"Distribution of Duration by Subscribtion Status",
                  labels= {'duration' : 'Duration' , 'count' : 'Count' , 'y' : 'Status'},
                  color_discrete_map={'no': '#EF553B', 'yes': '#00CC96'},
                  marginal_x= 'box',
                  template= 'plotly_dark').show()

# **Feature Engineering**

In [17]:
x = df.drop(columns=['y', 'duration'])

y = df['y'].map({'yes' : 1, 'no' : 0})

In [18]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.month_map = {
            'jan': 1, 'feb': 2, 'mar': 3,
            'apr': 4, 'may': 5, 'jun': 6,
            'jul': 7, 'aug': 8, 'sep': 9,
            'oct': 10, 'nov': 11, 'dec': 12
        }

    def fit(self, x):
        return self    
    
    def transform(self, x):
        data = x.copy()

        data['pdays'] = data['pdays'].replace(-1, 0)
        data['contacted_before'] = (data['pdays'] != 0).astype(int)

        data['month_num'] = data['month'].map(self.month_map)

        data['month_sin'] = np.sin(2 * np.pi * data['month_num'] / 12)
        data['month_cos'] = np.cos(2 * np.pi * data['month_num'] / 12)

        data = data.drop(columns=['month', 'pdays'])

        return data


In [19]:
feature_engineering = FeatureEngineering()

x = feature_engineering.fit_transform(x)

In [20]:
fig8 = px.scatter(
        x, 
        x="month_cos", 
        y="month_sin", 
        color="month_num",
        title="Cyclic Representation",
        labels={"month_cos": "X-axis (Cosine)", "month_sin": "Y-axis (Sine)", "month_num" : 'Month'},
        hover_data={"month_num": True, "month_sin": ":.3f", "month_cos": ":.3f"},
        template='plotly_dark').update_layout(
    
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        xaxis_range=[-1.2, 1.2],
        yaxis_range=[-1.2, 1.2]
        ).show()

# **Preprocessing**

In [21]:
num_f = ['age','campaign','previous','month_num','month_sin','month_cos']

balance_col = ['balance']

cat_f = ['job','marital','education','default','housing','loan','contact','poutcome']

num_p = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

balance_p = Pipeline(steps=[
    ('balance', PowerTransformer(method= 'yeo-johnson'))
])

cat_p = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_p, num_f),
    ('bal', balance_p, balance_col),
    ('cat', cat_p, cat_f)
])

# **Train & Test Split**

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state= 42, stratify= y)

# **Training Models**

In [23]:
models = {
    "Logistic Regression" : LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest" : RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    "SVC" : SVC(class_weight='balanced', probability=True, random_state=42)
}


cv = StratifiedKFold(n_splits= 3, shuffle= True, random_state=42)
scoring = ['f1', 'precision', 'recall', 'roc_auc']

for name, model in models.items():
    print(f"Training {name}...")
    full_pip = Pipeline(steps=[
        ('preprocessor', preprocessor),
        (f'{name}' , model)
    ])

    cv_score = cross_validate(full_pip, x_train, y_train, cv= cv, scoring= scoring)

    print(f"ROC-AUC:   {np.mean(cv_score['test_roc_auc']):.3f}")
    print(f"F1-Score:  {np.mean(cv_score['test_f1']):.3f}")
    print(f"Precision: {np.mean(cv_score['test_precision']):.3f}")
    print(f"Recall:    {np.mean(cv_score['test_recall']):.3f}\n")

Training Logistic Regression...
ROC-AUC:   0.746
F1-Score:  0.340
Precision: 0.230
Recall:    0.656

Training Random Forest...
ROC-AUC:   0.753
F1-Score:  0.297
Precision: 0.550
Recall:    0.204

Training SVC...
ROC-AUC:   0.767
F1-Score:  0.411
Precision: 0.316
Recall:    0.587



**SVC is our winner as it got almost the highest F1-Score and ROC-AUC scores**

# **Fine Tuning**

In [24]:
svc_pip = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('SVC', SVC(class_weight= 'balanced', probability=True, random_state= 42))
])

svc_params = {"SVC__kernel": ["rbf"], "SVC__gamma": ["scale", "auto"], "SVC__degree": [2, 3]}

rs = RandomizedSearchCV(
    estimator= svc_pip,
    param_distributions= svc_params,
    n_iter= 5, 
    cv= cv,
    scoring={'roc_auc': 'roc_auc', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall'},
    refit='f1',
    n_jobs= -1
)

rs.fit(x_train, y_train)

print(f"Best Params: {rs.best_params_}")

Best Params: {'SVC__kernel': 'rbf', 'SVC__gamma': 'scale', 'SVC__degree': 2}


In [25]:
rs.best_estimator_

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['age', 'campaign',
                                                   'previous', 'month_num',
                                                   'month_sin', 'month_cos']),
                                                 ('bal',
                                                  Pipeline(steps=[('balance',
                                                                   PowerTransformer())]),
                                                  ['balance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'poutcome'])])),
                ('SVC',
                 SVC(class_weight='balanced', degree=2, probability=True,
                     random_state=42))])

In [26]:
df_metrics = pd.DataFrame({
    "Metric" : ["ROC-AUC", "F1", "Precision", "Recall"],
    "Score" : [
        rs.cv_results_['mean_test_roc_auc'][rs.best_index_],
        rs.cv_results_['mean_test_f1'][rs.best_index_],
        rs.cv_results_['mean_test_precision'][rs.best_index_],
        rs.cv_results_['mean_test_recall'][rs.best_index_]
    ]
})

fig9 = px.bar(
    df_metrics,
    x= 'Metric',
    y='Score',
    color= 'Metric', 
    hover_data= ['Score'],
    title= "Best Scores After Tuning",
    template='plotly_dark').update_layout(showlegend= False).show()

# **Model Evaluation**

In [27]:
y_pred = rs.predict(x_test)

cl = classification_report(y_test, y_pred)
print(f"Classification Report: \n{cl}")

cm = confusion_matrix(y_test, y_pred)

fig10= px.imshow(cm,
                zmax=cm.max(),
                zmin=0,
                text_auto=True,
                aspect="auto",
                title="Confusion Matrix",
                color_continuous_scale="Blues",
                template="plotly_dark").update_layout(width= 800, 
                                                      height= 800, 
                                                      xaxis_title= 'What the Model Guessed', 
                                                      yaxis_title= 'What Actually Happened').update_xaxes(tickvals=[0,1],
                                                                                                          ticktext= ["Predicted No", "Predicted Yes"]).update_yaxes(tickvals= [0,1],
                                                                                                                                                                    ticktext= ["Actual No", "Actual Yes"]).show()

Classification Report: 
              precision    recall  f1-score   support

           0       0.94      0.83      0.88      7985
           1       0.32      0.60      0.42      1058

    accuracy                           0.80      9043
   macro avg       0.63      0.71      0.65      9043
weighted avg       0.87      0.80      0.83      9043



# **Feature Importance**

In [28]:
fimp = permutation_importance(
    rs, x_test, y_test, n_repeats= 5, scoring= 'f1', n_jobs= -1
)

imp_df = pd.DataFrame({
    "Features" : x_test.columns,
    "Importance" : fimp.importances_mean
}).sort_values(by='Importance', ascending=True).tail(10)

fig11 = px.bar(imp_df,
              x='Importance', 
              y='Features',
              color='Importance',
              orientation='h',
              title='Top 10 Most Important Features',
              color_continuous_scale='Mint',
              template='plotly_dark').show()

# **Saving the model**

In [29]:
jl.dump(rs, "SVC.pkl")

['SVC.pkl']